In [ ]:
import pandas as pd
import numpy as np

import statsmodels.api as sm

In [ ]:
#  both functions below return a df of daily rv
#  column names of df are different between the two dfs though
#  calc_intraday_rv uses short-term intervals throughout the day
#  calc_daily_rv uses daily intervals

def calc_intraday_rv(
                        df,
                        timestamp_col="timestamp",
                        price_col="close",
                        annualization_days=365,
                    ):
    """
    Input:
        df with intraday bars, e.g. 1-min or 5-min BTC bars

    Output:
        daily realized variance and realized volatility
    """

    df = df.copy()

    df[timestamp_col] = pd.to_datetime(df[timestamp_col])
    df = df.sort_values(timestamp_col)

    # Calendar date for grouping
    df["date"] = df[timestamp_col].dt.date

    # Intraday log returns by taking difference of log prices (i.e., same as log(ratio of prices))
    df["log_price"] = np.log(df[price_col])
    df["intraday_return"] = df.groupby("date")["log_price"].diff()

    # Squared returns
    df["squared_return"] = df["intraday_return"] ** 2

    # Daily realized variance
    # group by date
    # .agg => compute these stats
    # rv = sum of squared return column
    # n_bars = number of price observations in day
    # timestamps are for data checking
    daily_df = (
                   df.groupby("date", as_index=False)
                   .agg(
                       rv=("squared_return", "sum"),
                       n_bars=("squared_return", "count"),
                       first_timestamp=(timestamp_col, "min"),
                       last_timestamp=(timestamp_col, "max"),
                       )
               )

    # Daily and annualized realized volatility
    daily_df["realized_vol_daily"] = np.sqrt(daily_df["rv"])
    daily_df["realized_vol_ann"] = daily_df["realized_vol_daily"] * np.sqrt(annualization_days)

    return daily_df

#--------------------------------------

def calc_daily_rv(df, price_col="close"):
    """
    Simple daily realized volatility proxy from close-to-close returns.
    Later we can replace this with intraday RV.
    """
    df = df.copy()
    df["log_return"] = np.log(df[price_col] / df[price_col].shift(1))
    df["rv"] = df["log_return"] ** 2
    return df


In [ ]:
def make_har_features(df, rv_col="rv", rv_w_days=5, rv_m_days=22):
    df = df.copy()

    # HAR inputs
    df["rv_d"] = df[rv_col]
    df["rv_w"] = df[rv_col].rolling(rv_w_days).mean()
    df["rv_m"] = df[rv_col].rolling(rv_m_days).mean()

    # predict next day's RV
    df["target_rv_next"] = df[rv_col].shift(-1)

    df = df.dropna()

    return df


def fit_har_model(df):
    x_cols = ["rv_d", "rv_w", "rv_m"]

    X = df[x_cols]
    y = df["target_rv_next"]

    X = sm.add_constant(X)

    model = sm.OLS(y, X).fit()

    return model


def predict_next_rv(model, latest_df):
    x_cols = ["rv_d", "rv_w", "rv_m"]

    X_latest = latest_df[x_cols].tail(1)
    X_latest = sm.add_constant(X_latest, has_constant="add")

    pred_rv = model.predict(X_latest).iloc[0]

    return pred_rv

In [ ]:
df = pd.read_csv("btc_daily.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date")

df = calc_daily_rv(df, price_col="close")
har_df = make_har_features(df)

model = fit_har_model(har_df)

print(model.summary())

next_rv = predict_next_rv(model, har_df)

next_vol_daily = np.sqrt(next_rv)
next_vol_annualized = next_vol_daily * np.sqrt(365)

print("Predicted next-day RV:", next_rv)
print("Predicted next-day daily vol:", next_vol_daily)
print("Predicted annualized vol:", next_vol_annualized)